## Purpose:
    This script refines LSTM-based route prediction results by correcting
    disconnected or implausible node transitions using the hexagon road network graph.
    The raw predicted route is first simplified into a set of survivor nodes, and
    then shortest paths between consecutive survivor nodes are inserted to construct
    a graph-connected route.

## Input:
    - Hexagon road network graph:
      ./data/new_hexagraph/hexa_network_with_road.gpickle
    - Raw LSTM prediction files:
      ./expected_findings/code8_prediction_LSTM/LSTM_{k}_by1.csv
      where k = 8, 12, and 16

## Output:
    - Graph-corrected LSTM prediction files:
      ./expected_findings/code9_refine_LSTM/LSTM_fix_{k}_by1.csv

## Main procedures:
    1. Load the hexagon road network graph.
    2. Read raw LSTM-predicted route sequences.
    3. Select survivor nodes from each raw predicted route based on shortest-path distance.
    4. Connect consecutive survivor nodes using graph shortest paths.
    5. Save the corrected LSTM route prediction files.

In [1]:
import torch
import numpy as np
import pandas as pd
import networkx as nx
import pickle
from torch_geometric.utils import from_networkx
import os
import ast
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

## Refine model

In [2]:
def refine_path(G: nx.Graph, raw_path: list):
    """
    Refine a raw predicted route into a graph-connected route.

    Parameters:
        G:
            NetworkX graph representing the hexagon road network.
        raw_path:
            Raw predicted route sequence.

    Returns:
        A corrected route sequence where consecutive survivor nodes are connected
        by graph shortest paths.
    """
    survivors = [raw_path[0]]  # Always retain the start node.
    cur_idx   = 0
    n         = len(raw_path)

    # Step 1: Select survivor nodes from the raw predicted route.
    while cur_idx < n - 1:
        base = raw_path[cur_idx]
        found = False

        # Search forward from the current base node.
        for offset in range(1, n - cur_idx):
            idx = cur_idx + offset
            node = raw_path[idx]

            # Always retain the final destination node.
            if idx == n - 1:
                survivors.append(node)
                cur_idx = idx
                found = True
                break

            # Compute the shortest-path distance between the base node and candidate node.
            try:
                dist = nx.shortest_path_length(G, base, node)
            except nx.NetworkXNoPath:
                continue

            # Retain the candidate node if it is reachable within the corresponding offset.
            if dist <= offset:
                survivors.append(node)
                cur_idx = idx
                found = True
                break

        if not found:
            survivors.append(raw_path[-1])
            break

    # Step 2: Connect consecutive survivor nodes using graph shortest paths.
    final_path = [survivors[0]]
    for u, v in zip(survivors, survivors[1:]):
        segment = nx.shortest_path(G, u, v)
        final_path.extend(segment[1:]) # Exclude the first node to avoid duplication.

    return final_path


## Main execution

In [5]:
# Load the graph
GRAPH_PATH = './data/new_hexagraph/hexa_network_with_road.gpickle'
with open(GRAPH_PATH, 'rb') as f:
    G = pickle.load(f)
data = from_networkx(G)
num_nodes = data.num_nodes
node_features_cpu = torch.eye(num_nodes, dtype=torch.float32)  # 1-hot
edge_index_cpu = data.edge_index.to(torch.long)

node_feat_dim = node_features_cpu.size(1)

In [6]:
for k in [8, 12, 16]:
    pred_df = pd.read_csv(f'./expected_findings/code8_prediction_LSTM/LSTM_{k}_by1.csv')
    for i in range(0, len(pred_df)):
        pred_path = ast.literal_eval(pred_df.iloc[i, 1])
        pred_path = refine_path(G, pred_path)
        pred_df.iloc[i,1] = pred_path
    pred_df.to_csv(f'./expected_findings/code9_refine_LSTM/LSTM_fix_{k}_by1.csv', index=False)